In [ ]:
from pathlib import Path
from frameit.core.settings_class import SimulationConfig
from frameit.core.runner import FrameitRunner

import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import xarray as xr
import math

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_batsirai.yml"))
conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/arome_oper_belna.yml"))

# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_chido.yml"))
# conf = SimulationConfig.from_yaml_with_model_preset(Path("../../configs_ktests/mnh_ianos.yml"))

conf.printID()

runner = FrameitRunner(conf)
runner.run()

In [ ]:
runner.track.latitude

In [ ]:
proj = ccrs.PlateCarree()
fig, ax = plt.subplots(
    figsize=(15, 15),
    subplot_kw=dict(projection=proj)          # <-- GeoAxes !
)
ax.add_feature(cfeature.LAKES,
               facecolor='#A6CEE3')          # lacs bleu pâle
ax.coastlines(resolution='10m', linewidth=0.8, color='black')
ax.add_feature(cfeature.BORDERS, linewidth=0.5, edgecolor='#555555')

line,=ax.plot(
        runner.track.longitude,
        runner.track.latitude,
        color='k',
        ls='-',
        linewidth=3.0,
        transform=proj       
    )
ax.set_xlim(40,70)
ax.set_ylim(-30,-5)

In [ ]:
track_ds=runner.ds_tracker['wind_pressure']['surface']
conf=runner.conf
print(track_ds)

cx_name: str = "cx",
cy_name: str = "cy",

tdim = "time"
lonv_name =  getattr(conf, "name_longitude", None)
latv_name =  getattr(conf, "name_latitude", None)

if lonv_name is None or latv_name is None:
    print("[track_kinematics] conf missing name_longitude/name_latitude, skip.")
    
if tdim not in track_ds.dims:
    print("[track_kinematics] track_ds missing time dim %r, skip.", tdim)

if (cx_name not in track_ds) or (cy_name not in track_ds):
    print("[track_kinematics] track_ds missing %r/%r, skip.", cx_name, cy_name)
    
nt = int(track_ds.sizes.get(tdim, 0)) ; print(nt)

lon_src = track_ds[lonv_name]
lat_src = track_ds[latv_name]


# Align lon/lat time axis to track time if present
if tdim in lon_src.dims and tdim in track_ds.coords:
    try:
        print('here')
        lon_src = lon_src.sel({tdim: track_ds[tdim]})
        lat_src = lat_src.sel({tdim: track_ds[tdim]})
    except Exception:
        pass

# Identify horizontal dims as the last two non-time dims
non_time_dims = [d for d in lon_src.dims if d != tdim]
if len(non_time_dims) < 2:
    print(
        "[track_kinematics] lon/lat do not look like 2D fields, dims=%s, skip.", lon_src.dims
    )


In [ ]:





# Align lon/lat time axis to track time if present
if tdim in lon_src.dims and tdim in track_ds.coords:
    try:
        lon_src = lon_src.sel({tdim: track_ds[tdim]})
        lat_src = lat_src.sel({tdim: track_ds[tdim]})
    except Exception:
        pass

# Identify horizontal dims as the last two non-time dims
non_time_dims = [d for d in lon_src.dims if d != tdim]
if len(non_time_dims) < 2:
    print(
        "[track_kinematics] lon/lat do not look like 2D fields, dims=%s, skip.", lon_src.dims
    )

In [ ]:
lon_src.dims

In [ ]:
non_time_dims

In [ ]:
from pyproj import Geod
from collections.abc import Mapping, Sequence

def make_tracking_dataset(
    ds_tree: Mapping[str, Mapping[str, xr.Dataset]],
    method: str,
) -> xr.Dataset:
    """
    À partir d'un dictionnaire hiérarchique de la forme:

        ds_tree[method][vertical_group] -> xr.Dataset

    construit un xarray.Dataset plat pour la méthode donnée en fusionnant
    tous les sous-Datasets de cette méthode.
    """
    try:
        method_group = ds_tree[method]
    except KeyError as exc:
        raise ValueError(f"Tracking method {method!r} not found in ds_tree") from exc

    datasets = []
    for vert_group, ds in method_group.items():
        if not isinstance(ds, xr.Dataset):
            raise TypeError(
                f"For method {method!r}, vertical group {vert_group!r} "
                f"should be an xarray.Dataset, got {type(ds)}"
            )
        datasets.append(ds)

    if not datasets:
        raise ValueError(f"No Dataset found for method {method!r} in ds_tree")

    ds_flat = xr.merge(datasets, compat="no_conflicts")
    return ds_flat



def _norm_az_deg(az_deg: np.ndarray) -> np.ndarray:
    """Normalize azimuth to [0, 360)."""
    az = np.asarray(az_deg, dtype=float)
    return (az % 360.0 + 360.0) % 360.0


def enrich_track_with_kinematics(
    track_ds: xr.Dataset,
    *,
    ds_flat: xr.Dataset,
    conf,
    ellps: str = "WGS84",
    time_dim: str | None = None,
    lon_name: str | None = None,
    lat_name: str | None = None,
    cx_name: str = "cx",
    cy_name: str = "cy",
    min_step_m: float = 1.0,
) -> xr.Dataset:
    """
    Post-process a tracker output to add lon/lat and simple kinematics.

    Inputs
    ------
    track_ds:
        Must contain cx(time) and cy(time) indices (gridpoint indices).
    ds_flat:
        Flattened dataset used for tracking, must contain lon/lat fields as 2D
        (y,x) or 3D (time,y,x).
    conf:
        Used to retrieve tracking_method, name_time_dim, name_longitude, name_latitude.

    Outputs added to track_ds
    -------------------------
    lon(time), lat(time):
        Center lon/lat sampled at (cy,cx).
    heading_deg(time):
        Forward geodesic azimuth from (t) to (t+1), degrees, clockwise from North.
        The last value is copied from the previous step.
    d_m(time):
        Step distance (t -> t+1), meters, last copied.
    speed_ms(time):
        d_m / dt in m s-1, last copied.

    Behavior
    --------
    - If tracking_method == "fixed_box", returns track_ds unchanged.
    - Permissive: if required fields are missing, logs a warning and returns unchanged.
    - If step distance < min_step_m, heading is kept from previous valid value when possible.
    """

    tracking_method = str(getattr(conf, "tracking_method", "")).lower()
    if tracking_method == "fixed_box":
        return track_ds

    tdim = "time"
    lonv_name = lon_name or getattr(conf, "name_longitude", None)
    latv_name = lat_name or getattr(conf, "name_latitude", None)

    if lonv_name is None or latv_name is None:
        print("[track_kinematics] conf missing name_longitude/name_latitude, skip.")
        return track_ds

    if tdim not in track_ds.dims:
        print("[track_kinematics] track_ds missing time dim %r, skip.", tdim)
        return track_ds

    if (cx_name not in track_ds) or (cy_name not in track_ds):
        print("[track_kinematics] track_ds missing %r/%r, skip.", cx_name, cy_name)
        return track_ds

    if (lonv_name not in ds_flat) or (latv_name not in ds_flat):
        print(
            "[track_kinematics] ds_flat missing lon/lat (%s/%s), skip.", lonv_name, latv_name
        )
        return track_ds

    nt = int(track_ds.sizes.get(tdim, 0))
    if nt < 2:
        return track_ds

    lon_src = ds_flat[lonv_name]
    lat_src = ds_flat[latv_name]

    # Align lon/lat time axis to track time if present
    if tdim in lon_src.dims and tdim in track_ds.coords:
        try:
            lon_src = lon_src.sel({tdim: track_ds[tdim]})
            lat_src = lat_src.sel({tdim: track_ds[tdim]})
        except Exception:
            pass

    # Identify horizontal dims as the last two non-time dims
    non_time_dims = [d for d in lon_src.dims if d != tdim]
    if len(non_time_dims) < 2:
        print(
            "[track_kinematics] lon/lat do not look like 2D fields, dims=%s, skip.", lon_src.dims
        )
        return track_ds

    ydim, xdim = non_time_dims[-2], non_time_dims[-1]

    cy = track_ds[cy_name]
    cx = track_ds[cx_name]

    try:
        lon_c = lon_src.isel({ydim: cy, xdim: cx})
        lat_c = lat_src.isel({ydim: cy, xdim: cx})
    except Exception as exc:
        print(
            "[track_kinematics] failed to sample lon/lat at (cy,cx) using (%s,%s): %s",
            ydim,
            xdim,
            exc,
        )
        return track_ds

    # Ensure lon_c/lat_c have time dimension (broadcast if lon/lat were time-invariant)
    if tdim not in lon_c.dims:
        lon_c = lon_c.expand_dims({tdim: track_ds[tdim]})
        lat_c = lat_c.expand_dims({tdim: track_ds[tdim]})

    lon_arr = np.asarray(lon_c.values, dtype=float)
    lat_arr = np.asarray(lat_c.values, dtype=float)

    geod = Geod(ellps=ellps)
    az12, _, dist_m = geod.inv(lon_arr[:-1], lat_arr[:-1], lon_arr[1:], lat_arr[1:])
    az12 = _norm_az_deg(np.asarray(az12, dtype=float))
    dist_m = np.asarray(dist_m, dtype=float)

    # dt in seconds
    tvals = track_ds[tdim].values.astype("datetime64[s]")
    dt_s = np.diff(tvals).astype("timedelta64[s]").astype(float)
    dt_s = np.where(dt_s > 0, dt_s, np.nan)

    speed = dist_m / dt_s

    heading_deg = np.full((nt,), np.nan, dtype=float)
    d_m = np.full((nt,), np.nan, dtype=float)
    speed_ms = np.full((nt,), np.nan, dtype=float)

    heading_deg[:-1] = az12
    d_m[:-1] = dist_m / 1e3
    speed_ms[:-1] = speed

    # Copy last
    heading_deg[-1] = heading_deg[-2]
    d_m[-1] = d_m[-2]
    speed_ms[-1] = speed_ms[-2]

    # Guard against near-stationary steps, keep previous heading when possible
    if min_step_m is not None and float(min_step_m) > 0.0:
        bad = d_m < float(min_step_m)
        if np.any(bad):
            for i in range(1, nt):
                if bad[i]:
                    heading_deg[i] = heading_deg[i - 1]

    out = track_ds.copy()
    out["lon"] = lon_c.astype(float)
    out["lat"] = lat_c.astype(float)
    out["heading_deg"] = xr.DataArray(heading_deg, dims=(tdim,))
    out["dist"] = xr.DataArray(d_m, dims=(tdim,))
    out["speed"] = xr.DataArray(speed_ms, dims=(tdim,))

    out["lon"].attrs.update(units="degrees_east", long_name="cyclone center longitude")
    out["lat"].attrs.update(units="degrees_north", long_name="cyclone center latitude")
    out["heading_deg"].attrs.update(
        units="degree", long_name="cyclone heading (azimuth, clockwise from north)"
    )
    out["dist"].attrs.update(units="km", long_name="step distance (forward)")
    out["speed"].attrs.update(units="m s-1", long_name="step speed (forward)")

    return out


In [ ]:
ds_flat = make_tracking_dataset(runner.ds_tracker, self.tracker.name)
track = enrich_track_with_kinematics(
    track,
    ds_flat=ds_flat,
    conf=self.conf,
    cx_name="cx",
    cy_name="cy",
)

In [ ]:
track = enrich_track_with_kinematics(
            runner.ds_tracker,
            ds_flat=ds_flat,
            conf=self.conf,
            cx_name="cx",
            cy_name="cy",
        )